# HR Workforce Analytics — Python (Pandas) Analysis

Reproduces the same KPIs and breakdowns as the Excel / SQL / Power BI / Tableau versions of this project.

**Requirements:** `pip install pandas openpyxl matplotlib`


## 1. Load & Clean

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

FILE = "HR_Employee_Data_Raw.xlsx"   # change path if needed

df = pd.read_excel(FILE)

print("Rows loaded:", len(df))
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Duplicate Employee_IDs:", df["Employee_ID"].duplicated().sum())

df["Join_Date"] = pd.to_datetime(df["Join_Date"])
df["Month"] = df["Join_Date"].dt.strftime("%b")
df["Month_Num"] = df["Join_Date"].dt.month


## 2. Headline KPIs

In [ ]:
kpis = {
    "Total Employees": len(df),
    "Total Attrition": (df["Attrition"] == "Yes").sum(),
    "Attrition Rate": round((df["Attrition"] == "Yes").mean(), 4),
    "Avg Monthly Salary": round(df["Monthly_Salary"].mean(), 0),
    "Avg Annual CTC": round(df["Annual_CTC"].mean(), 0),
    "Avg Age": round(df["Age"].mean(), 1),
    "Avg Performance Rating": round(df["Performance_Rating"].mean(), 2),
    "Avg Experience (yrs)": round(df["Experience_Years"].mean(), 1),
}

for k, v in kpis.items():
    print(f"{k}: {v:,}" if isinstance(v, (int, float)) else f"{k}: {v}")


## 3. Breakdowns

In [ ]:
monthly_hires = df.groupby("Month_Num").agg(
    new_hires=("Employee_ID", "count"),
    avg_starting_salary=("Monthly_Salary", "mean"),
).sort_index()
monthly_hires.index = df.groupby("Month_Num")["Month"].first().sort_index()

dept_summary = df.groupby("Department").agg(
    headcount=("Employee_ID", "count"),
    avg_ctc=("Annual_CTC", "mean"),
    avg_rating=("Performance_Rating", "mean"),
).sort_values("headcount", ascending=False)

state_headcount = df["State"].value_counts()

level_summary = (
    df.groupby("Designation_Level")
    .agg(headcount=("Employee_ID", "count"), avg_salary=("Monthly_Salary", "mean"))
    .sort_values("avg_salary", ascending=False)
)

employment_split = df["Employment_Type"].value_counts()

manager_summary = (
    df.groupby("Manager_Name")
    .agg(team_size=("Employee_ID", "count"), avg_rating=("Performance_Rating", "mean"))
    .sort_values("team_size", ascending=False)
)

attrition_by_dept = df.groupby("Department").agg(
    attrition_count=("Attrition", lambda x: (x == "Yes").sum()),
    headcount=("Employee_ID", "count"),
)
attrition_by_dept["attrition_rate"] = attrition_by_dept["attrition_count"] / attrition_by_dept["headcount"]
attrition_by_dept = attrition_by_dept.sort_values("attrition_rate", ascending=False)

rating_dist = df["Performance_Rating"].value_counts().sort_index()

gender_pay = df.groupby("Gender").agg(
    headcount=("Employee_ID", "count"),
    avg_salary=("Monthly_Salary", "mean"),
    avg_rating=("Performance_Rating", "mean"),
)

age_bins = [21, 30, 40, 50, 58]
age_labels = ["22-30", "31-40", "41-50", "51-58"]
df["Age_Group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels)
age_group_summary = df.groupby("Age_Group", observed=True).agg(
    headcount=("Employee_ID", "count"), avg_salary=("Monthly_Salary", "mean")
)

exp_bins = [-1, 5, 10, 20, 30]
exp_labels = ["0-5 yrs", "6-10 yrs", "11-20 yrs", "21-30 yrs"]
df["Exp_Band"] = pd.cut(df["Experience_Years"], bins=exp_bins, labels=exp_labels)
exp_band_summary = df.groupby("Exp_Band", observed=True).agg(
    headcount=("Employee_ID", "count"), avg_salary=("Monthly_Salary", "mean")
)

attrition_compare = df.groupby("Attrition").agg(
    headcount=("Employee_ID", "count"),
    avg_rating=("Performance_Rating", "mean"),
    avg_leaves=("Leaves_Taken", "mean"),
    avg_overtime=("Overtime_Hours", "mean"),
)


In [ ]:
print("--- Department Summary ---")
dept_summary

In [ ]:
print("--- State Headcount ---")
state_headcount

In [ ]:
print("--- Designation Level — Avg Salary ---")
level_summary

In [ ]:
print("--- Manager Summary (top 5) ---")
manager_summary.head()

In [ ]:
print("--- Attrition by Department (sorted by rate) ---")
attrition_by_dept

In [ ]:
print("--- Gender Pay Comparison ---")
gender_pay

In [ ]:
print("--- Attrited vs Retained ---")
attrition_compare

## 4. Charts

In [ ]:
plt.style.use("default")
NAVY, ORANGE, TEAL, GOLD = "#0B1E3D", "#ED7D31", "#2E8B7C", "#C9A227"

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.patch.set_facecolor("white")

# Monthly hiring trend
axes[0, 0].plot(monthly_hires.index, monthly_hires["new_hires"], marker="o", color=TEAL, linewidth=2)
axes[0, 0].set_title("Monthly New Hires Trend")
axes[0, 0].set_ylabel("New Hires")
axes[0, 0].tick_params(axis="x", rotation=45)

# Department headcount
axes[0, 1].barh(dept_summary.index, dept_summary["headcount"], color=ORANGE)
axes[0, 1].set_title("Department Headcount")
axes[0, 1].invert_yaxis()

# Employment type donut
axes[1, 0].pie(
    employment_split.values, labels=employment_split.index, autopct="%1.0f%%",
    colors=[NAVY, ORANGE, TEAL], wedgeprops=dict(width=0.4),
)
axes[1, 0].set_title("Employment Type Split")

# Attrition rate by department
axes[1, 1].bar(attrition_by_dept.index, attrition_by_dept["attrition_rate"] * 100, color=GOLD)
axes[1, 1].set_title("Attrition Rate by Department (%)")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("hr_python_analysis_charts.png", dpi=130)
plt.show()


## 5. Export Summary to Excel (optional)

In [ ]:
with pd.ExcelWriter("hr_python_analysis_output.xlsx") as writer:
    pd.DataFrame([kpis]).to_excel(writer, sheet_name="KPIs", index=False)
    dept_summary.to_excel(writer, sheet_name="Department")
    state_headcount.to_excel(writer, sheet_name="State")
    level_summary.to_excel(writer, sheet_name="Designation_Level")
    manager_summary.to_excel(writer, sheet_name="Manager")
    attrition_by_dept.to_excel(writer, sheet_name="Attrition_by_Dept")
    monthly_hires.to_excel(writer, sheet_name="Monthly_Hires")

print("Summary workbook saved to hr_python_analysis_output.xlsx")
